In [ ]:
import os
import logging

from typing import List
from datasets import load_dataset

from haystack.utils import Secret
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack_integrations.components.generators.amazon_bedrock import AmazonBedrockGenerator

from haystack.dataclasses import ChatMessage


from haystack.components.builders import PromptBuilder
from haystack.components.routers import ConditionalRouter
from duckduckgo_api_haystack import DuckduckgoApiWebSearch
from haystack.components.joiners import BranchJoiner


from haystack.utils import Secret
from haystack import Pipeline
from haystack_experimental.core import AsyncPipeline
from milvus_haystack import MilvusDocumentStore
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack_integrations.components.embedders.fastembed import FastembedSparseDocumentEmbedder
from haystack_integrations.components.generators.amazon_bedrock import AmazonBedrockGenerator


from haystack_integrations.components.embedders.optimum import (
    OptimumTextEmbedder,
    OptimumDocumentEmbedder,
    OptimumEmbedderPooling,
    OptimumEmbedderOptimizationConfig,
    OptimumEmbedderOptimizationMode,
)


from milvus_haystack import MilvusEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersTextEmbedder

from haystack.components.joiners.document_joiner import DocumentJoiner
from haystack.components.builders import AnswerBuilder
from haystack.components.joiners import AnswerJoiner

from haystack_experimental.chat_message_stores.in_memory import InMemoryChatMessageStore
from haystack_experimental.chat_message_stores.distributed import DistributedChatMessageStore
from haystack_experimental.components.retrievers import ChatMessageRetriever
from haystack_experimental.components.writers import ChatMessageWriter



from ml.prompt_meta import (rag_prompt_template_conditional,
                             rag_prompt_template_full,
                             rag_prompt_web,
                             rag_prompt_curriculum,
                             prompt_template_web_fallback, 
                             web_fallback_routes
                      )
from ..tools.async_tools import ConcurrentGenerators, ConcurrentRetrievers, ListJoiner

ImportError: attempted relative import with no known parent package

In [20]:
logging.basicConfig(level=logging.WARN)

# Imports

In [17]:
import os
import logging

from typing import List
from datasets import load_dataset

from haystack.utils import Secret
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack_integrations.components.generators.amazon_bedrock import AmazonBedrockGenerator

from haystack.dataclasses import ChatMessage


from haystack.components.builders import PromptBuilder
from haystack.components.routers import ConditionalRouter
from duckduckgo_api_haystack import DuckduckgoApiWebSearch
from haystack.components.joiners import BranchJoiner


from haystack.utils import Secret
from haystack import Pipeline
from haystack_experimental.core import AsyncPipeline
from milvus_haystack import MilvusDocumentStore
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack_integrations.components.embedders.fastembed import FastembedSparseDocumentEmbedder
from haystack_integrations.components.generators.amazon_bedrock import AmazonBedrockGenerator


from haystack_integrations.components.embedders.optimum import (
    OptimumTextEmbedder,
    OptimumDocumentEmbedder,
    OptimumEmbedderPooling,
    OptimumEmbedderOptimizationConfig,
    OptimumEmbedderOptimizationMode,
)


from milvus_haystack import MilvusEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersTextEmbedder

from haystack.components.joiners.document_joiner import DocumentJoiner
from haystack.components.builders import AnswerBuilder
from haystack.components.joiners import AnswerJoiner

from haystack_experimental.chat_message_stores.in_memory import InMemoryChatMessageStore
from haystack_experimental.chat_message_stores.distributed import DistributedChatMessageStore
from haystack_experimental.components.retrievers import ChatMessageRetriever
from haystack_experimental.components.writers import ChatMessageWriter

from ml.prompt_meta import (rag_prompt_template_conditional,
                             rag_prompt_template_full,
                             rag_prompt_web,
                             rag_prompt_curriculum,
                             prompt_template_web_fallback, 
                             web_fallback_routes)


# Models

In [9]:
import os
from haystack.utils import Secret
from haystack.components.generators import OpenAIGenerator
from haystack_integrations.components.generators.amazon_bedrock import AmazonBedrockGenerator
from haystack_integrations.components.generators.anthropic import AnthropicGenerator
from ml.models import vllm_config, openai_config, build_model



vllm_config = {"api_key": Secret.from_token("VLLM-PLACEHOLDER-API-KEY"),
               "model": "Ensure to use your own knowledge with the context where reasonable",
               "api_base_url":"http://localhost:8000/v1",
               "generation_kwargs": {"max_tokens": 512, "temperature": 0.2}
                 }

openai_config = {"api_key": Secret.from_token(os.getenv("OPEN_AI_KEY")),
               "model": 'gpt-4o-mini',
               "api_base_url":None,
               "generation_kwargs": {"temperature": 0.2}
                 }

In [23]:
from transformers import AutoModel

model_name = "sentence-transformers/all-mpnet-base-v2"
model = AutoModel.from_pretrained(model_name, cache_dir="/mnt/nfs/models")


c:\Users\Richard\anaconda3\envs\ai-agent\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\mnt\nfs\models\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [24]:

# Define the model path
model_path = "/mnt/nfs/models"

# Check if the directory exists
if not os.path.exists(model_path):
    # If it doesn't exist, create it
    os.makedirs(model_path)
    print(f"Created directory: {model_path}")
else:
    print(f"Directory already exists: {model_path}")

Directory already exists: /mnt/nfs/models


# Milvus

In [14]:
document_store = MilvusDocumentStore(
        collection_name='textbook_example_openai',
        connection_args={"uri": os.environ.get('HAYSTACK_CLUSTER_URI'), 
                        "token": os.environ.get('HAYSTACK_CLUSTER_TOKEN'),
                        "secure": True
                        },
        drop_old=False,
    )

conv_store = MilvusDocumentStore(
        collection_name='conv_cache',
        connection_args={"uri": os.environ.get('HAYSTACK_CLUSTER_URI'), 
                        "token": os.environ.get('HAYSTACK_CLUSTER_TOKEN'),
                        "secure": True
                        },
        drop_old=True,
        consistency_level="Strong"
    )

Some secret values are not encrypted. Please use `Secret` class to encrypt them. The best way to implement it is to use `Secret.from_env` to load from environment variables. For example:
from haystack.utils import Secret
token = Secret.from_env('YOUR_TOKEN_ENV_VAR_NAME')
Some secret values are not encrypted. Please use `Secret` class to encrypt them. The best way to implement it is to use `Secret.from_env` to load from environment variables. For example:
from haystack.utils import Secret
token = Secret.from_env('YOUR_TOKEN_ENV_VAR_NAME')


# Pipeline

In [10]:
def build_model(config, backend="openai"):

  assert backend in ["openai", "bedrock", "anthropic", "vllm"], "Invalid backend"

  if backend == "openai":
     generator = OpenAIGenerator(
      api_key=config['api_key'],
      model=config['model'],
      api_base_url=config['api_base_url'],
      generation_kwargs=config['generation_kwargs'],
      system_prompt="Make sure you stick to your prompt template")
     
     generator_con = OpenAIGenerator(
      api_key=config['api_key'],
      model=config['model'],
      api_base_url=config['api_base_url'],
      generation_kwargs=config['generation_kwargs'],
      system_prompt="Make sure you stick to your prompt template")

  elif backend == "bedrock":
    generator = AmazonBedrockGenerator(model="anthropic.claude-3-5-sonnet-20240620-v1:0",
                                       aws_access_key_id=Secret.from_token(os.getenv("BEAM_ACCESS_AWS")),
                                       aws_secret_access_key=Secret.from_token(os.getenv("BEAM_SECRET_AWS")),
                                     
                                       )
    generator_con = AmazonBedrockGenerator(model="anthropic.claude-3-5-sonnet-20240620-v1:0",
                                       aws_access_key_id=Secret.from_token(os.getenv("BEAM_ACCESS_AWS")),
                                       aws_secret_access_key=Secret.from_token(os.getenv("BEAM_SECRET_AWS")),
                                      


                                       )

  elif backend == "anthropic":
    generator = AnthropicGenerator(Secret.from_env_var("ANTHROPIC_API_KEY"),
                                   model="claude-3-5-haiku-20241022")
    generator_con = AnthropicGenerator(Secret.from_env_var("ANTHROPIC_API_KEY"),
                                       model="claude-3-5-haiku-20241022")


  elif backend == "vllm":
    generator = OpenAIGenerator(
    api_key=Secret.from_token("VLLM-PLACEHOLDER-API-KEY"),
    model="mistralai/Mistral-7B-Instruct-v0.1",
    api_base_url="http://localhost:8000/v1",
    generation_kwargs = {"max_tokens": 512}
    ) 


  return generator, generator_con

In [19]:
openai=False
onnx=False
backend="openai"
ragk=5
webk=5
hybrid_retrieval=True
model_path = "/mnt/nfs/models"

generator, generator_con = build_model(openai_config, backend=backend)

text_embedder = SentenceTransformersTextEmbedder(model=model_path)
document_embedder = SentenceTransformersDocumentEmbedder(model=model_path)

In [25]:
retriever = MilvusEmbeddingRetriever(document_store=document_store, top_k=ragk)

prompt_builder = PromptBuilder(template=rag_prompt_web,
                                 variables=["documents", "memories", "query"], required_variables=["documents", "memories", "query"])
prompt_builder_c = PromptBuilder(template=rag_prompt_curriculum, 
                                   variables=["documents", "memories", "query"], required_variables=["documents", "memories", "query"]
                                   )
router = ConditionalRouter(web_fallback_routes)
websearch = DuckduckgoApiWebSearch(top_k=webk,  backend="lite")
prompt_builder_after_websearch = PromptBuilder(template=prompt_template_web_fallback)
prompt_joiner  = BranchJoiner(str)

  #memory
memory_store = DistributedChatMessageStore(document_store=conv_store, 
                                                document_embedder=document_embedder
                                                )
  
memory_retriever = ChatMessageRetriever(memory_store)
memory_writer = ChatMessageWriter(memory_store)

No sentence-transformers model found with name /mnt/nfs/models. Creating a new one with mean pooling.


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /mnt/nfs/models.

In [ ]:
doc_pipe.add_component("text_embedder", text_embedder)
doc_pipe.add_component("retriever", retriever)
doc_pipe.connect("text_embedder.embedding", "retriever.query_embedding")

In [ ]:
doc_pipe_web.add_component("websearch", websearch)

In [ ]:
mem_pipe.add_component("memory_retriever", memory_retriever)

## Milvus

In [ ]:
## To connect to database
from pymilvus import connections
import os
from pymilvus import Collection

# Retrieve environment variables
HAYSTACK_CLUSTER_URI = os.environ.get("HAYSTACK_CLUSTER_URI_DEV")
HAYSTACK_CLUSTER_TOKEN = os.environ.get("HAYSTACK_CLUSTER_TOKEN_DEV")

# Establish connection to Milvus
connections.connect(
    alias="default",
    uri=HAYSTACK_CLUSTER_URI,
    token=HAYSTACK_CLUSTER_TOKEN,
    secure=True  # Set to True if your Milvus instance uses TLS/SSL
)

# Verify the connection
if connections.has_connection("default"):
    print("Successfully connected to Milvus.")
else:
    print("Failed to connect to Milvus.")

Successfully connected to Milvus.


In [93]:
# List all existing collections
collections = utility.list_collections()
print("Available Collections:", collections)

Available Collections: ['textbook_example_openai']


In [95]:
# Specify the collection name
collection_name = "textbook_example_openai"
collection = Collection(collection_name)

# Print schema details
print("Collection Schema:", collection.schema)
print("Indexed Fields:", collection.indexes)
print("Partitions:", collection.partitions)


Collection Schema: {'auto_id': False, 'description': '', 'fields': [{'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'name': 'start', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'name': 'end', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}}, {'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535}, 'is_primary': True, 'auto_id': False}, {'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False}
Indexed Fields: [<pymilvus.orm.index.Index object at 0x0000022815ED96C0>]
Partitions: [{"name":"_default","collection_name":"textbook_example_openai","description":""}]


In [96]:

# Extract schema details
schema_dict = {
    "auto_id": collection.schema.auto_id,
    "enable_dynamic_field": collection.schema.enable_dynamic_field,
    "fields": [
        {
            "name": field.name,
            "type": str(field.dtype),
            "max_length": field.params.get("max_length", "N/A"),
            "is_primary": getattr(field, "is_primary", False),
            "auto_id": getattr(field, "auto_id", False),
        }
        for field in collection.schema.fields
    ],
}

# Pretty-print schema
formatted_schema = json.dumps(schema_dict, indent=4)


In [50]:
print(formatted_schema)

{
    "auto_id": false,
    "enable_dynamic_field": false,
    "fields": [
        {
            "name": "title",
            "type": "DataType.VARCHAR",
            "max_length": 65535,
            "is_primary": false,
            "auto_id": false
        },
        {
            "name": "start",
            "type": "DataType.VARCHAR",
            "max_length": 65535,
            "is_primary": false,
            "auto_id": false
        },
        {
            "name": "end",
            "type": "DataType.VARCHAR",
            "max_length": 65535,
            "is_primary": false,
            "auto_id": false
        },
        {
            "name": "text",
            "type": "DataType.VARCHAR",
            "max_length": 65535,
            "is_primary": false,
            "auto_id": false
        },
        {
            "name": "id",
            "type": "DataType.VARCHAR",
            "max_length": 65535,
            "is_primary": true,
            "auto_id": false
        },
       

In [98]:
#Example of quierying the collection
# Query to get the first 2 documents 

query_results = collection.query(
    expr="id != ''",  # Ensures we retrieve a valid document
    output_fields=["id","start","end", "title", "text", "vector"],
    limit=2  # Get only the first document
)




In [100]:
query_results[0]

{'id': '0001699ff3699435e2324c682bbed3b502188c32f877774ad380b9f890cd7241',
 'start': '01:07:54.925',
 'end': '01:12:56.045',
 'title': 'Thursday Lecture (MLE-B4) 2023-12-07 MLOps #2 - Linux',
 'text': "So every data that you store right in, in AWS, it makes sure that it is fault tolerant. Suppose, uh, the data is stored on a particular, um, it is stored in a particular region, right? And for whatever reason, you know, it fails, uh, it has to be made sure that you don't lose data. So there is some kind of a replication factor that is baked in, right? Um, that says that, um, it can always retrieve the data from elsewhere, right? And, uh, all of these are basically site liability engineering, uh, that have taken place, uh, or that AWS makes sure that it takes care to make sure that your data is not lost and it's always available for you. Okay? Uh, typically the replication factor is three, and it's made sure that, uh, the same data is not on the same rack. Uh, what if the rack, uh, rack, 

# Haystack Pipelines